# TaxiWise — Exploratory Data Analysis
**NYC Yellow Taxi 2025 | ניתוח נתונים חקרני**

קבצים:
- `yellow_taxi_2025.csv` — 30,000 נסיעות ב-2025
- `data/taxi_zone_lookup.csv` — 265 אזורי מונית ב-NYC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

## 1. טעינת נתונים

In [ ]:
df = pd.read_csv('yellow_taxi_2025.csv', parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])
zones = pd.read_csv('data/taxi_zone_lookup.csv')

print(f'נסיעות: {len(df):,}')
print(f'עמודות: {df.shape[1]}')
print(f'תקופה: {df.tpep_pickup_datetime.min().date()} עד {df.tpep_pickup_datetime.max().date()}')
df.head(3)

## 2. סקירה כללית

In [ ]:
df.info()

In [ ]:
numeric_cols = ['passenger_count','trip_distance','fare_amount','tip_amount',
                'total_amount','trip_duration_min','congestion_surcharge','tolls_amount']
df[numeric_cols].describe().round(2)

## 3. ערכים חסרים

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print('אין ערכים חסרים בנתונים')
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing.plot(kind='bar', ax=ax, color='tomato')
    ax.set_title('ערכים חסרים לפי עמודה')
    ax.set_ylabel('מספר שורות')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    print(missing.to_string())

## 4. פיצול עמודות זמן

In [ ]:
df['pickup_hour']    = df.tpep_pickup_datetime.dt.hour
df['pickup_dow']     = df.tpep_pickup_datetime.dt.day_of_week   # 0=Mon
df['pickup_month']   = df.tpep_pickup_datetime.dt.month
df['pickup_day_name']= df.tpep_pickup_datetime.dt.day_name()
print('עמודות זמן נוצרו בהצלחה')

## 5. ניתוח זמני

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# נסיעות לפי שעה
hourly = df.groupby('pickup_hour').size()
axes[0].bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
axes[0].set_title('נסיעות לפי שעת איסוף')
axes[0].set_xlabel('שעה')
axes[0].set_ylabel('מספר נסיעות')
axes[0].set_xticks(range(0, 24))

# נסיעות לפי יום בשבוע
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('pickup_day_name').size().reindex(day_order)
axes[1].bar(range(7), dow.values, color='coral', edgecolor='white')
axes[1].set_title('נסיעות לפי יום בשבוע')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['ב','ג','ד','ה','ו','ש','א'], fontsize=11)
axes[1].set_ylabel('מספר נסיעות')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: שעה × יום
pivot = df.pivot_table(index='pickup_hour', columns='pickup_dow', values='VendorID', aggfunc='count')
pivot.columns = ['ב','ג','ד','ה','ו','ש','א']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, ax=ax,
            cbar_kws={'label': 'מספר נסיעות'})
ax.set_title('צפיפות נסיעות: שעה × יום בשבוע')
ax.set_xlabel('יום בשבוע')
ax.set_ylabel('שעה')
plt.tight_layout()
plt.show()

## 6. ניתוח גיאוגרפי — בורו ואזורים

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
                           ['pickup_borough', 'dropoff_borough'],
                           ['איסוף לפי בורו', 'הורדה לפי בורו']):
    counts = df[col].value_counts()
    ax.barh(counts.index, counts.values, color='mediumseagreen', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('מספר נסיעות')
    for i, v in enumerate(counts.values):
        ax.text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 15 אזורי האיסוף המובילים
top_pu = df['pickup_zone'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
top_pu.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('15 אזורי האיסוף המובילים')
ax.set_xlabel('מספר נסיעות')
plt.tight_layout()
plt.show()

In [ ]:
# מטריצת זרימה בין-בורו
flow = df.groupby(['pickup_borough', 'dropoff_borough']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(flow, annot=True, fmt=',d', cmap='Blues',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'נסיעות'})
ax.set_title('מטריצת זרימת נסיעות בין בורואים')
ax.set_xlabel('בורו יעד')
ax.set_ylabel('בורו מוצא')
plt.tight_layout()
plt.show()

## 7. ניתוח תעריפים

In [ ]:
# הסרת outliers קיצוניים לצורך הצגה
df_clean = df[(df.fare_amount > 0) & (df.fare_amount < df.fare_amount.quantile(0.99)) &
              (df.total_amount > 0) & (df.total_amount < df.total_amount.quantile(0.99))]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title, color in zip(axes,
    ['fare_amount', 'tip_amount', 'total_amount'],
    ['תעריף בסיס ($)', 'טיפ ($)', 'סכום כולל ($)'],
    ['steelblue', 'goldenrod', 'coral']):
    ax.hist(df_clean[col], bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df_clean[col].median(), color='black', linestyle='--', linewidth=1.2, label=f'חציון: ${df_clean[col].median():.1f}')
    ax.set_title(title)
    ax.set_xlabel('$')
    ax.set_ylabel('תדירות')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# תעריף ממוצע לפי בורו
fare_by_borough = df_clean.groupby('pickup_borough')['total_amount'].agg(['mean','median']).round(2)
fare_by_borough.columns = ['ממוצע', 'חציון']
print('סכום כולל לפי בורו איסוף ($):')
fare_by_borough.sort_values('ממוצע', ascending=False)

In [ ]:
# אחוז טיפ מהתעריף לפי שעה
df_clean2 = df_clean[df_clean.payment_type == 1].copy()  # רק כרטיס אשראי
df_clean2['tip_pct'] = df_clean2.tip_amount / df_clean2.fare_amount * 100

tip_hour = df_clean2.groupby('pickup_hour')['tip_pct'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tip_hour.index, tip_hour.values, marker='o', color='darkorange', linewidth=2)
ax.fill_between(tip_hour.index, tip_hour.values, alpha=0.2, color='darkorange')
ax.set_title('אחוז טיפ ממוצע לפי שעה (תשלום בכרטיס אשראי)')
ax.set_xlabel('שעה')
ax.set_ylabel('טיפ (%)')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 8. ניתוח מרחק ומשך נסיעה

In [ ]:
df_trips = df[(df.trip_distance > 0) & (df.trip_distance < df.trip_distance.quantile(0.99)) &
              (df.trip_duration_min > 0) & (df.trip_duration_min < df.trip_duration_min.quantile(0.99))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_trips.trip_distance, bins=60, color='teal', edgecolor='white', alpha=0.85)
axes[0].axvline(df_trips.trip_distance.median(), color='red', linestyle='--', linewidth=1.2,
                label=f'חציון: {df_trips.trip_distance.median():.1f} מייל')
axes[0].set_title('התפלגות מרחק נסיעה')
axes[0].set_xlabel('מרחק (מייל)')
axes[0].legend()

axes[1].hist(df_trips.trip_duration_min, bins=60, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1].axvline(df_trips.trip_duration_min.median(), color='red', linestyle='--', linewidth=1.2,
                label=f'חציון: {df_trips.trip_duration_min.median():.1f} דקות')
axes[1].set_title('התפלגות משך נסיעה')
axes[1].set_xlabel('דקות')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# מרחק vs. תעריף
sample = df_trips.sample(min(3000, len(df_trips)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(sample.trip_distance, sample.fare_amount,
                c=sample.pickup_hour, cmap='plasma', alpha=0.5, s=10)
plt.colorbar(sc, ax=ax, label='שעת איסוף')
ax.set_title('מרחק נסיעה מול תעריף בסיס (צבע = שעה)')
ax.set_xlabel('מרחק (מייל)')
ax.set_ylabel('תעריף ($)')
plt.tight_layout()
plt.show()

In [ ]:
# מהירות ממוצעת לפי שעה
df_speed = df_trips.copy()
df_speed['speed_mph'] = df_speed.trip_distance / (df_speed.trip_duration_min / 60)
df_speed = df_speed[df_speed.speed_mph < 80]

speed_hour = df_speed.groupby('pickup_hour')['speed_mph'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(speed_hour.index, speed_hour.values, marker='s', color='teal', linewidth=2)
ax.fill_between(speed_hour.index, speed_hour.values, alpha=0.15, color='teal')
ax.set_title('מהירות נסיעה ממוצעת לפי שעה (mph)')
ax.set_xlabel('שעה')
ax.set_ylabel('mph')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 9. סוג תשלום ומספר נוסעים

In [ ]:
payment_labels = {1: 'כרטיס אשראי', 2: 'מזומן', 3: 'חיוב ישיר', 4: 'מחלוקת', 5: 'לא ידוע', 6: 'נסיעה מבוטלת'}
ptype = df['payment_type'].map(payment_labels).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(ptype.values, labels=ptype.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(ptype)), startangle=140)
axes[0].set_title('פילוג סוג תשלום')

pcount = df[df.passenger_count.between(1, 6)]['passenger_count'].value_counts().sort_index()
axes[1].bar(pcount.index.astype(int), pcount.values, color='orchid', edgecolor='white')
axes[1].set_title('פילוג מספר נוסעים')
axes[1].set_xlabel('נוסעים')
axes[1].set_ylabel('נסיעות')
axes[1].set_xticks(range(1, 7))

plt.tight_layout()
plt.show()

## 10. ניתוח אזורי מונית (taxi_zone_lookup)

In [ ]:
print('אזורים לפי בורו:')
print(zones.groupby('Borough')['LocationID'].count().sort_values(ascending=False).to_string())
print(f'\nסוגי שירות: {zones.service_zone.unique()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

boro_count = zones.groupby('Borough').size().sort_values()
axes[0].barh(boro_count.index, boro_count.values, color='steelblue')
axes[0].set_title('מספר אזורים לפי בורו')
axes[0].set_xlabel('אזורים')

szone = zones.groupby('service_zone').size().sort_values(ascending=False)
axes[1].bar(szone.index, szone.values, color=['steelblue','coral','mediumseagreen','goldenrod'][:len(szone)])
axes[1].set_title('פילוג סוגי אזור שירות')
axes[1].set_ylabel('אזורים')
plt.xticks(rotation=20, ha='right')

plt.tight_layout()
plt.show()

## 11. ביצועים לפי אזור — Top/Bottom

In [ ]:
zone_stats = df_clean.groupby('pickup_zone').agg(
    נסיעות=('VendorID', 'count'),
    הכנסה_ממוצעת=('total_amount', 'mean'),
    מרחק_ממוצע=('trip_distance', 'mean'),
    משך_ממוצע=('trip_duration_min', 'mean')
).round(2).sort_values('נסיעות', ascending=False)

print('10 אזורי האיסוף המובילים:')
zone_stats.head(10)

In [ ]:
# השוואת הכנסה ממוצעת — top 10 לפי הכנסה
top_revenue = zone_stats.nlargest(10, 'הכנסה_ממוצעת')

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_revenue.index, top_revenue['הכנסה_ממוצעת'], color='goldenrod')
ax.set_title('10 אזורים עם הכנסה ממוצעת גבוהה ביותר ($)')
ax.set_xlabel('הכנסה ממוצעת ($)')
for bar, val in zip(bars, top_revenue['הכנסה_ממוצעת']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'${val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 12. מתאמים

In [ ]:
corr_cols = ['trip_distance','trip_duration_min','fare_amount','tip_amount',
             'total_amount','passenger_count','pickup_hour']
corr = df_clean[corr_cols].corr()

hebrew_labels = ['מרחק','משך','תעריף','טיפ','סכום כולל','נוסעים','שעה']
corr.index = hebrew_labels
corr.columns = hebrew_labels

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('מטריצת מתאמים')
plt.tight_layout()
plt.show()

## 13. סיכום ממצאים

In [ ]:
peak_hour = hourly.idxmax()
quiet_hour = hourly.idxmin()
top_borough = df['pickup_borough'].value_counts().idxmax()
median_fare = df_clean['total_amount'].median()
median_dist = df_trips['trip_distance'].median()
median_dur  = df_trips['trip_duration_min'].median()
credit_pct  = (df['payment_type'] == 1).mean() * 100

print('=' * 45)
print('        סיכום ממצאי EDA — TaxiWise')
print('=' * 45)
print(f'  נסיעות בסה"כ         : {len(df):,}')
print(f'  שעת שיא              : {peak_hour}:00  ({hourly[peak_hour]:,} נסיעות)')
print(f'  שעה שקטה             : {quiet_hour}:00  ({hourly[quiet_hour]:,} נסיעות)')
print(f'  בורו מוביל           : {top_borough}')
print(f'  חציון סכום כולל      : ${median_fare:.2f}')
print(f'  חציון מרחק           : {median_dist:.1f} מייל')
print(f'  חציון משך            : {median_dur:.1f} דקות')
print(f'  תשלום בכרטיס אשראי   : {credit_pct:.1f}%')
print('=' * 45)